<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Custom Local Disk Sizes

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook covers:** Every FABRIC node includes a **local disk** -- a virtual disk that holds the operating system and provides space for your experiment data. By default, nodes get a 10 GB disk, but many experiments need more space. This notebook shows how to customize local disk sizes and demonstrates how the actual disk allocation works.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Understand the role of **local disk** in FABRIC nodes
2. Request different local disk sizes using the `disk` parameter
3. Understand how disk size **hints** are rounded up to instance types
4. Verify actual disk space inside a running VM using `df -h`

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you should:

1. Complete the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Successfully run the [Hello, FABRIC](../hello_fabric/hello_fabric.ipynb) notebook

**Tip:** Your project must have the `VM.NoLimit` permission tag to allocate VMs with more than 10 GB of disk. Contact your project lead if you need this permission.

</div>

## Background: Local Disk vs. Other Storage

FABRIC offers three types of storage, each suited for different use cases:


The **local disk** is the simplest option: it is built into every VM and requires no extra configuration. The `disk` parameter in `add_node()` is a **minimum hint** -- FABRIC rounds up to the nearest instance type. For example:

| Requested Disk | Actual Instance Type | Actual Disk |
|:---:|:---:|:---:|
| 10 GB | `fabric.c2.m8.d10` | 10 GB |
| 50 GB | `fabric.c4.m16.d100` | 100 GB |
| 100 GB | `fabric.c4.m16.d100` | 100 GB |
| 500 GB | `fabric.c16.m64.d500` | 500 GB |

## What We're Building

In this notebook we will create multiple nodes with different local disk sizes (10 GB to 500 GB).

<img src="./figs/slice_topology.png" width="40%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

---

## Step 2: Create Nodes with Different Disk Sizes

We will create five nodes in a single slice, each with a different `disk` value (including one with the default). This lets us compare the actual allocated disk sizes side by side.

<div class="fab-warning">

**Tip:** The `disk` value is only a hint. The actual disk size depends on which instance type FABRIC selects. Larger disk requests may also result in more cores and RAM because instance types bundle all three together.

</div>

In [ ]:
slice_name = 'MySlice'

# Create a new slice
slice = fablib.new_slice(slice_name)

# Add five nodes with varying disk sizes:
#   NodeDefault -- uses the default disk size (10 GB)
#   Node10      -- explicitly requests 10 GB
#   Node50      -- requests 50 GB (will be rounded up to 100 GB)
#   Node100     -- requests 100 GB
#   Node500     -- requests 500 GB
slice.add_node(name='NodeDefault')
slice.add_node(name='Node10', disk=10)
slice.add_node(name='Node50', disk=50)
slice.add_node(name='Node100', disk=100)
slice.add_node(name='Node500', disk=500)

# Submit the slice request and wait for all nodes to be provisioned
slice.submit()

---

## Step 3: Observe the Actual Disk Sizes

Now let's SSH into each node and run `df -h` to see the actual disk space available. Notice how the requested sizes are rounded up to the nearest instance type.

In [ ]:
# Loop through all nodes and check their actual disk sizes
for node in slice.get_nodes():
    print(f"\n{node.get_name()}\n")
    # 'df -h' shows disk usage in human-readable format
    # Look at the root filesystem (/) to see the total local disk size
    node.execute('df -h')

<div class="fab-success">

**What to look for:** Compare the `Size` column for the root filesystem (`/`) across all five nodes. You should see that:
- `NodeDefault` and `Node10` have the same small disk (both use the default 10 GB instance type)
- `Node50` and `Node100` likely have the same disk (both round up to the 100 GB instance type)
- `Node500` has the largest disk

</div>

---

## Step 4: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running unnecessarily prevents other researchers from using those resources.

</div>

In [ ]:
# Delete the slice and release all resources
fablib.delete_slice("MySlice")

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `PDP Authorization check failed` with `VM.NoLimit` message | Project lacks permission for large VMs | Contact your project lead to request the `VM.NoLimit` tag |
| Actual disk is larger than requested | FABRIC rounds up to the nearest instance type | This is expected behavior -- you always get at least what you asked for |
| Slice fails with resource error | Site does not have enough capacity for all five nodes | Try a different site or create fewer nodes |
| `df -h` shows very little free space despite large disk | OS and packages consume some disk space | The full disk is allocated but the OS uses a portion of it |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `fablib.delete_slice(name)` | Delete a slice by name | [delete_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.delete_slice) |
| `slice.add_node(name, disk)` | Add a node with a custom disk size hint | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `slice.get_nodes()` | Get list of all node objects in the slice | [get_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_nodes) |
| `node.get_name()` | Get the node's name | [get_name](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.get_name) |
| `node.execute(command)` | Execute a shell command on the node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |

## What's Next?

Now that you understand local disk storage, explore these related notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **NVMe Storage** | [basic_nvme_devices](../basic_nvme_devices/basic_nvme_devices.ipynb) | Add fast NVMe storage devices (1 TB) to your nodes |
| **Persistent Storage** | [persistent_storage](../persistent_storage/persistent_storage.ipynb) | Attach project-level persistent storage that survives slice deletion |
| **Customizing Nodes** | [customizing_nodes](../customizing_nodes/customizing_nodes.ipynb) | Set site, cores, RAM, and OS image |
| **Running Commands** | [execute_commands](../ssh_to_nodes/execute_commands.ipynb) | Execute commands, capture output, use threads |